# South African ASR on Colab, without Google Drive

Colab's disk is wiped when the session ends, so everything worth keeping is stored in
**private Hugging Face repos** under your account:

| Repo | Holds |
|---|---|
| `<you>/sa-asr-results` (dataset) | baseline and comparison results |
| `<you>/sa-asr-whisper-lora` (model) | training checkpoints and, at the end, the finished model |

**Before you start**
1. `sa-asr.zip` (the project) is on your laptop.
2. You accepted the terms at https://huggingface.co/datasets/dsfsi-anv/za-african-next-voices
3. You have a Hugging Face token with **Write** permission (Settings → Access Tokens).
4. Runtime → Change runtime type → a **GPU** (T4 on the free plan; A100/L4 if you have them).
   Settings adjust to the GPU automatically.

**Every session (including after a disconnect):** run cells **1 → 4**, then continue with the step
you were on. Finished work is pulled back from Hugging Face and skipped.

## 1. Settings
Change `HF_USER` to your Hugging Face username, then run.

In [ ]:
HF_USER = "your-hf-username"   # <- change this to your Hugging Face username

RESULTS_REPO = f"{HF_USER}/sa-asr-results"
MODEL_REPO = f"{HF_USER}/sa-asr-whisper-lora"

# Adjust to the GPU: batch size for training, and which baseline models are worth the time.
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU (not CPU or TPU)"
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 2**30
if gpu_gb > 30:   # A100 / H100
    BATCH = "--batch-size 32 --grad-accum 1"
    EVAL_SAMPLES = 200
    BASELINE_MODELS = ["whisper-large-v3-turbo", "whisper-large-v3", "mms-1b-all", "dsfsi-anv-multilingual-turbo"]
else:             # T4 / L4: fewer test utterances, and skip whisper-large-v3 (slowest, not the one we fine-tune)
    BATCH = "--batch-size 8 --grad-accum 4"
    EVAL_SAMPLES = 100
    BASELINE_MODELS = ["whisper-large-v3-turbo", "dsfsi-anv-multilingual-turbo", "mms-1b-all"]
BASELINE_ARG = " ".join(BASELINE_MODELS)
print(f"GPU: {gpu.name} ({gpu_gb:.0f} GB)")
print(f"Training batch: {BATCH}")
print(f"Baseline: {EVAL_SAMPLES} test utterances per set; models {BASELINE_MODELS}")

## 2. Upload the project
A file picker appears: choose `sa-asr.zip` from your laptop. Skipped if the project is already here.

In [ ]:
import os, zipfile
if not os.path.exists('/content/sa-asr/pyproject.toml'):
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall('/content')
%cd /content/sa-asr

## 3. Install (about 2 minutes)
If Colab asks to **restart the session**, accept, then run cells 1–3 again.

In [ ]:
!pip install -q -e ".[dev]"
# Colab preinstalls torchao 0.10, which recent PEFT refuses to work with. We don't use it.
!pip uninstall -q -y torchao

## 4. Log in to Hugging Face and check access

A box appears under the cell: paste your **Write** token (it shows as dots), press **Enter**.
The check at the end must print `User: …` and two OK lines; if one fails, the message says what to do.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass("Paste your Hugging Face WRITE token and press Enter: "))

# Runs as separate processes, exactly like the scripts below, so it checks what they will see.
!python -c "import sa_asr; from huggingface_hub import HfApi; print('User:', HfApi().whoami()['name'])"
!python -c "import sa_asr; from huggingface_hub import auth_check; auth_check('dsfsi-anv/za-african-next-voices', repo_type='dataset'); print('Swivuriso access: OK')" 2>/dev/null || echo "Swivuriso access: FAILED -> accept the terms at https://huggingface.co/datasets/dsfsi-anv/za-african-next-voices (as the same user), then rerun this cell"
!python -c "import sa_asr; from huggingface_hub import HfApi; HfApi().create_repo('{RESULTS_REPO}', repo_type='dataset', private=True, exist_ok=True); print('Write access: OK')" 2>/dev/null || echo "Write access: FAILED -> the token needs Write permission, and HF_USER in cell 1 must be your username"

## 5. (Optional) Smoke test: only needed once

Checks the whole pipeline in about 5 minutes. Once it has passed (all tests pass, and
`Saved …outputs/smoke/final` at the end), **skip this cell** in later sessions.

In [ ]:
!python -m pytest -q
!python scripts/evaluate.py --config configs/eval_smoke.yaml
!python scripts/train.py --config configs/train_smoke.yaml

## 6. Baseline

Scores existing models on held-out test sets. The first ~30 minutes are spent sampling test audio.
Everything is saved to Hugging Face **as soon as it exists**: each sampled test set and each
finished result (`Pushed …` lines). After a disconnect or when the free GPU allowance runs out,
come back later, run cells 1–4 and this cell again: it pulls what is done and continues from there.
On a free T4 this may take two sessions.

In [ ]:
!python scripts/evaluate.py --config configs/eval_baseline.yaml --models {BASELINE_ARG} --max-samples {EVAL_SAMPLES} --sync-repo {RESULTS_REPO}

### Baseline table
Paste this table to Claude to decide the training setup.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/baseline/summary.md', encoding='utf-8').read()))

## 7. Fine-tune

Checkpoints are pushed to Hugging Face every 500 steps. The commands always include `--resume`:
on a first run they start fresh, and after a disconnect they continue from the last pushed checkpoint.

### 7a. Trial first
isiZulu and isiXhosa, 500 steps (about 30–60 minutes on a T4). Note two things for Claude:
- the **seconds per step** in the progress bar (e.g. `4.8s/it`)
- the `eval_wer_…` lines at the end, which should be well below the baseline for those languages

In [ ]:
!python scripts/train.py --config configs/train_lora.yaml {BATCH} --max-steps 500 --langs zul xho --output-dir outputs/trial --hub-repo {HF_USER}/sa-asr-trial --resume

### 7b. Full run
All 11 languages, 6,000 steps. Several hours, and on a T4 likely more than one session.
Keep the tab open. If Colab disconnects or runs out of GPU time, come back later, run cells 1–4,
and run this cell again: it continues where it stopped.

In [ ]:
!python scripts/train.py --config configs/train_lora.yaml {BATCH} --hub-repo {MODEL_REPO} --resume

## 8. Compare with the baseline

Scores the finished model (straight from `MODEL_REPO`) on the same test sets and adds it to the table.

In [ ]:
import yaml
from IPython.display import Markdown, display
cfg = yaml.safe_load(open('configs/eval_baseline.yaml'))
cfg['models'].append({'name': 'sa-turbo-lora', 'type': 'whisper', 'model': MODEL_REPO, 'language_mode': 'proxy'})
yaml.safe_dump(cfg, open('configs/eval_compare.yaml', 'w'), sort_keys=False)

!python scripts/evaluate.py --config configs/eval_compare.yaml --models sa-turbo-lora --max-samples {EVAL_SAMPLES} --sync-repo {RESULTS_REPO}
display(Markdown(open('results/baseline/summary.md', encoding='utf-8').read()))

## 9. Use it in the app on your laptop

In the app's sidebar, enter the model as `<your-username>/sa-asr-whisper-lora`. It is downloaded from
Hugging Face automatically (log in on the laptop first; see the README).